# Film Factory — GPU Image Animation

Animates static images into 4-second parallax clips for the Film Factory pipeline.

**Methods (in order of quality):**
1. **Stable Video Diffusion (SVD-XT)** — Diffusion-based animation, most cinematic
2. **AnimateDiff + MotionLoRA** — Fast, stylistically consistent
3. **DepthFlow (fallback)** — CPU-compatible depth warp, no GPU needed

**Output:** Upload MP4s to Google Drive → set `MOTION_CLIPS_DIR` below → re-run compositor

**Runtime:** A100 recommended. T4 works for AnimateDiff. SVD needs ≥16GB VRAM.

In [ ]:
# @title 1. Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

# ── Configure these paths ──────────────────────────────────────────────
# Where to read input images FROM (upload your outputs/images/ folder here)
INPUT_IMAGES_DIR = '/content/drive/MyDrive/film_factory/images'

# Where to write finished motion clips TO (compositor reads from here)
MOTION_CLIPS_DIR = '/content/drive/MyDrive/film_factory/motion_clips'

# Animation method: 'svd' | 'animatediff' | 'depthflow'
METHOD = 'svd'

# Clip settings
CLIP_DURATION = 4      # seconds
OUTPUT_FPS    = 24
OUTPUT_W      = 1920
OUTPUT_H      = 1080
# ──────────────────────────────────────────────────────────────────────

import os
os.makedirs(MOTION_CLIPS_DIR, exist_ok=True)
print(f'Input:  {INPUT_IMAGES_DIR}')
print(f'Output: {MOTION_CLIPS_DIR}')
print(f'Method: {METHOD}')

In [ ]:
# @title 2. Install dependencies
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Common
pip('diffusers>=0.27', 'transformers>=4.38', 'accelerate>=0.28',
    'opencv-python-headless', 'Pillow', 'imageio[ffmpeg]',
    'huggingface_hub', 'safetensors')

if METHOD == 'animatediff':
    pip('peft')

if METHOD == 'depthflow':
    pip('depthflow')

print('Dependencies installed.')

In [ ]:
# @title 3a. Stable Video Diffusion (SVD-XT) — best quality
# Requires A100 or V100 (16GB+ VRAM). Generates 25 frames (≈4s at 6fps → interpolated to 24fps)

if METHOD == 'svd':
    import torch
    from diffusers import StableVideoDiffusionPipeline
    from diffusers.utils import load_image, export_to_video
    from PIL import Image
    import numpy as np
    import os, cv2
    from pathlib import Path

    print('Loading SVD-XT pipeline...')
    pipe = StableVideoDiffusionPipeline.from_pretrained(
        'stabilityai/stable-video-diffusion-img2vid-xt',
        torch_dtype=torch.float16,
        variant='fp16',
    )
    pipe = pipe.to('cuda')
    pipe.enable_model_cpu_offload()
    print('SVD-XT pipeline ready.')

    def animate_svd(image_path: str, out_path: str) -> bool:
        """Generate 4s motion clip from a single image using SVD-XT."""
        try:
            img = Image.open(image_path).convert('RGB').resize((1024, 576))

            generator = torch.manual_seed(42)
            frames = pipe(
                img,
                decode_chunk_size=8,
                generator=generator,
                motion_bucket_id=127,   # 0=static, 255=max motion
                noise_aug_strength=0.02,
                num_frames=25,
            ).frames[0]

            # Export 25 frames → video at 6fps, then speed up to match 24fps duration
            tmp_path = out_path.replace('.mp4', '_raw.mp4')
            export_to_video(frames, tmp_path, fps=6)

            # Upscale to 1920x1080 and set to exactly CLIP_DURATION seconds at OUTPUT_FPS
            cmd = [
                'ffmpeg', '-y', '-i', tmp_path,
                '-vf', f'scale={OUTPUT_W}:{OUTPUT_H}:flags=lanczos,setsar=1',
                '-t', str(CLIP_DURATION),
                '-r', str(OUTPUT_FPS),
                '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                '-preset', 'veryfast', '-b:v', '6M',
                out_path,
            ]
            subprocess.run(cmd, capture_output=True)
            os.remove(tmp_path)
            return Path(out_path).exists()
        except Exception as e:
            print(f'SVD failed for {image_path}: {e}')
            return False

    print('animate_svd() ready.')

In [ ]:
# @title 3b. AnimateDiff + MotionLoRA — fast, consistent style
# Works on T4 (8GB). Camera motion LoRA: pan-left gives parallax feel for static scenes.

if METHOD == 'animatediff':
    import torch
    from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
    from diffusers.utils import export_to_gif
    from PIL import Image
    from pathlib import Path
    import subprocess, os

    print('Loading AnimateDiff + MotionLoRA...')
    adapter = MotionAdapter.from_pretrained(
        'guoyww/animatediff-motion-adapter-v1-5-2',
        torch_dtype=torch.float16,
    )
    # Realistic-vision base model (cinematic style)
    model_id = 'SG161222/Realistic_Vision_V5.1_noVAE'
    pipe = AnimateDiffPipeline.from_pretrained(
        model_id,
        motion_adapter=adapter,
        torch_dtype=torch.float16,
    )
    pipe.scheduler = DDIMScheduler.from_pretrained(
        model_id, subfolder='scheduler',
        beta_schedule='linear', clip_sample=False,
        timestep_spacing='linspace', steps_offset=1,
    )
    # Pan-left LoRA gives slow cinematic camera drift
    pipe.load_lora_weights(
        'guoyww/animatediff-motion-lora-pan-left',
        adapter_name='pan_left',
    )
    pipe.set_adapters(['pan_left'], [0.8])
    pipe.enable_vae_slicing()
    pipe = pipe.to('cuda')
    print('AnimateDiff pipeline ready.')

    def describe_image_for_animatediff(image_path: str) -> str:
        """Derive a safe prompt from the filename — no API call needed."""
        stem = Path(image_path).stem.replace('_', ' ').replace('-', ' ')
        return (
            f'cinematic scene, {stem}, dark atmosphere, high contrast, '
            'slow camera pan, shallow depth of field, film grain, 4K'
        )

    def animate_animatediff(image_path: str, out_path: str) -> bool:
        try:
            prompt = describe_image_for_animatediff(image_path)
            output = pipe(
                prompt=prompt,
                negative_prompt='blurry, low quality, watermark, text, logo',
                num_frames=16,
                guidance_scale=7.5,
                num_inference_steps=25,
                generator=torch.manual_seed(42),
                width=512, height=512,
            )
            frames = output.frames[0]

            # Save as GIF first, then convert + upscale
            gif_path = out_path.replace('.mp4', '.gif')
            export_to_gif(frames, gif_path)

            cmd = [
                'ffmpeg', '-y',
                '-stream_loop', '-1', '-t', str(CLIP_DURATION),
                '-i', gif_path,
                '-vf', f'scale={OUTPUT_W}:{OUTPUT_H}:flags=lanczos,setsar=1',
                '-r', str(OUTPUT_FPS),
                '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                '-preset', 'veryfast', '-b:v', '6M',
                out_path,
            ]
            subprocess.run(cmd, capture_output=True)
            os.remove(gif_path)
            return Path(out_path).exists()
        except Exception as e:
            print(f'AnimateDiff failed for {image_path}: {e}')
            return False

    print('animate_animatediff() ready.')

In [ ]:
# @title 3c. DepthFlow fallback — CPU-compatible depth warp

if METHOD == 'depthflow':
    from DepthFlow import DepthScene
    from pathlib import Path

    def animate_depthflow(image_path: str, out_path: str) -> bool:
        try:
            scene = DepthScene()
            scene.input(image=image_path)
            scene.main(output=out_path, time=CLIP_DURATION, fps=OUTPUT_FPS)
            return Path(out_path).exists()
        except Exception as e:
            print(f'DepthFlow failed: {e}')
            return False

    print('animate_depthflow() ready.')

In [ ]:
# @title 4. Run batch animation
import glob
from pathlib import Path

animate_fn = {
    'svd':         animate_svd,
    'animatediff': animate_animatediff,
    'depthflow':   animate_depthflow,
}[METHOD]

image_paths = sorted(
    glob.glob(f'{INPUT_IMAGES_DIR}/**/*.png', recursive=True) +
    glob.glob(f'{INPUT_IMAGES_DIR}/**/*.jpg', recursive=True)
)

print(f'Found {len(image_paths)} images. Starting animation...')

done = skipped = failed = 0

for img_path in image_paths:
    stem     = Path(img_path).stem
    out_path = f'{MOTION_CLIPS_DIR}/{stem}_motion.mp4'

    if Path(out_path).exists():
        print(f'  [skip] {stem}')
        skipped += 1
        continue

    print(f'  [anim] {stem}...')
    ok = animate_fn(img_path, out_path)
    if ok:
        print(f'         ✓ saved → {out_path}')
        done += 1
    else:
        print(f'         ✗ failed')
        failed += 1

print(f'\nDone: {done}  Skipped: {skipped}  Failed: {failed}')

## After running this notebook

1. The motion clips are saved in `MOTION_CLIPS_DIR` on your Drive.
2. Download them to your local machine at `outputs/motion_clips/`.
3. Re-run the compositor agent (Agent 6b) — it will pick up the pre-rendered clips automatically via `entry["motion_clip_path"]` without regenerating anything.

### Naming convention
The compositor looks for `outputs/motion_clips/{stem}_motion.mp4` where `{stem}` matches the source image filename without extension.  
Example: `ep1_scene2_diner.png` → `ep1_scene2_diner_motion.mp4`

### Gaussian Splatting (future upgrade)
True 3D Gaussian Splatting requires **multiple photos of the same location** (ideally 30-100 shots from different angles).  
For single-image 3D estimation, see **[RealDreamer](https://github.com/nlsde-safety-team/RealDreamer)** or **[Zero123++](https://github.com/SUDO-AI-3D/zero123plus)** which reconstruct a 3D scene from one image and can render novel viewpoints.  
A Gaussian Splatting notebook will be added here once those pipelines are integrated.